# MILPÍN AgTech — Modelo de Punto de Equilibrio con Churn
## ¿Cuántas hectáreas necesita MILPÍN para ser viable?

Modela la viabilidad del **Plan 1 (SaaS $120 MXN/ha/ciclo)** con una simulación de adopción que incluye churn mensual.

### Concepto central

| Fuerza | Qué es | Efecto |
|--------|--------|--------|
| **Adquisición** | % del mercado disponible que entra cada mes | Sube los ingresos |
| **Churn** | % de hectáreas activas que no renuevan cada mes | Baja los ingresos |

El equilibrio entre ambas define un **techo real** que puede estar muy por debajo del mercado disponible:

$$\text{Ha equilibrio} = \frac{\text{Techo} \times \text{Adquisición}}{\text{Adquisición} + \text{Churn}}$$

> **Ejemplo base:** techo=5,000 ha, adq=3%/mes, churn=5%/mes  
> Ha_eq = 5000 × 0.03 / 0.08 = **1,875 ha** (solo el 37.5% del mercado)

In [ ]:
# Instala si es necesario (descomenta y ejecuta una sola vez):
# !pip install matplotlib numpy pandas ipywidgets

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    from IPython.display import display
    WIDGETS_OK = True
except ImportError:
    WIDGETS_OK = False

print('Librerias cargadas')
print(f'  ipywidgets disponible: {WIDGETS_OK}')

## 1. Parámetros del modelo

Todo en un solo dict. Modifica aquí para cambiar los supuestos.

In [ ]:
PARAMS = {
    'techo_ha':    5_000,  # Ha maximas alcanzables (modulo 3 completo ~40,000 ha)
    'adquisicion': 0.03,   # Fraccion del mercado libre que entra por mes (3%)
    'churn':       0.05,   # Fraccion de ha activas que no renuevan por mes (5%)
    'precio_ha':   120,    # MXN por hectarea por ciclo (Plan 1)
    'ciclos_anio': 2,      # Ciclos agricolas/año (trigo invierno + maiz verano)
    'meses':       48,     # Horizonte de simulacion (4 años)
}

COSTOS = {
    'Stack actual (Supabase+Railway)': 19_360 / 12,   # ~1,613 MXN/mes
    'Azure empresarial (DR-041)':      292_680 / 12,  # ~24,390 MXN/mes
}

COLOR = {
    'pesimista': '#E24B4A', 'base': '#BA7517', 'optimista': '#1D9E75',
    'costo_a':   '#185FA5', 'costo_b': '#7F77DD', 'gris': '#AAAAAA',
}

for k, v in PARAMS.items():
    print(f'  {k}: {v}')

## 2. Funciones del modelo

In [ ]:
def simular_adopcion(techo_ha, adquisicion, churn, meses):
    """
    Simula hectareas activas mes a mes.

    Ecuacion discreta:
        nuevas   = adquisicion * (techo - ha)   <- boca a boca / ventas
        perdidas = churn * ha                   <- no renuevan
        ha_sig   = ha + nuevas - perdidas

    Equilibrio cuando nuevas == perdidas:
        ha_eq = techo * adq / (adq + churn)
    """
    ha, hist = 0.0, []
    for _ in range(meses):
        nuevas   = adquisicion * (techo_ha - ha)
        perdidas = churn * ha
        ha       = max(0.0, ha + nuevas - perdidas)
        hist.append(ha)
    return np.array(hist)


def calcular_equilibrio(techo_ha, adquisicion, churn):
    """Ha en estado estacionario (techo real del negocio)."""
    return techo_ha * adquisicion / (adquisicion + churn) if (adquisicion + churn) > 0 else 0.0


def ingreso_mensual(ha_arr, precio_ha, ciclos_anio):
    """MXN/mes del Plan 1."""
    return ha_arr * precio_ha * ciclos_anio / 12


def mes_breakeven(ingresos_arr, costo_mensual):
    """Primer mes donde ingreso >= costo. None si no ocurre."""
    idx = np.where(ingresos_arr >= costo_mensual)[0]
    return int(idx[0]) + 1 if len(idx) > 0 else None


# Test rapido
ha_test = simular_adopcion(5000, 0.03, 0.05, 48)
eq_test = calcular_equilibrio(5000, 0.03, 0.05)
print(f'Ha mes 1:   {ha_test[0]:.1f}')
print(f'Ha mes 12:  {ha_test[11]:.1f}')
print(f'Ha mes 48:  {ha_test[47]:.1f}')
print(f'Equilibrio: {eq_test:.1f} ha ({eq_test/5000*100:.1f}% del techo)')

## 3. Análisis por escenario

| Escenario | Adquisición | Churn |
|-----------|-------------|-------|
| Pesimista | Base × 0.6  | Base × 2 |
| Base      | Base        | Base     |
| Optimista | Base × 1.5  | Base × 0.5 |

In [ ]:
def construir_escenarios(p):
    configs = {
        'pesimista': (p['adquisicion'] * 0.6,  p['churn'] * 2.0),
        'base':      (p['adquisicion'],          p['churn']),
        'optimista': (p['adquisicion'] * 1.5,   p['churn'] * 0.5),
    }
    res = {}
    for nombre, (acq, ch) in configs.items():
        ha_arr  = simular_adopcion(p['techo_ha'], acq, ch, p['meses'])
        rev_arr = ingreso_mensual(ha_arr, p['precio_ha'], p['ciclos_anio'])
        eq_ha   = calcular_equilibrio(p['techo_ha'], acq, ch)
        res[nombre] = {
            'ha': ha_arr, 'rev': rev_arr, 'eq_ha': eq_ha,
            'eq_rev': eq_ha * p['precio_ha'] * p['ciclos_anio'] / 12,
            'acq': acq, 'ch': ch,
        }
    return res


esc = construir_escenarios(PARAMS)
filas = []
for nombre, d in esc.items():
    pct = d['eq_ha'] / PARAMS['techo_ha'] * 100
    fila = {
        'Escenario':            nombre.capitalize(),
        'Ha equilibrio':        f"{d['eq_ha']:,.0f}",
        '% del techo':          f"{pct:.1f}%",
        'Ingreso eq (MXN/mes)': f"${d['eq_rev']:,.0f}",
    }
    for stack, costo in COSTOS.items():
        be = mes_breakeven(d['rev'], costo)
        label = stack.split('(')[0].strip()
        fila[f'Breakeven {label}'] = f'Mes {be}' if be else 'No alcanza'
    filas.append(fila)

display(pd.DataFrame(filas))

## 4. Visualización estática

In [ ]:
def graficar(p):
    esc = construir_escenarios(p)
    meses_x = np.arange(1, p['meses'] + 1)
    lw = {'pesimista': 1.5, 'base': 2.2, 'optimista': 1.5}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')
    titulo = (
        'MILPIN - Modelo de Adopcion con Churn\n'
        f"adq={p['adquisicion']*100:.1f}%/mes | "
        f"churn={p['churn']*100:.1f}%/mes | "
        f"precio=${p['precio_ha']} MXN/ha/ciclo | "
        f"techo={p['techo_ha']:,} ha"
    )
    fig.suptitle(titulo, fontsize=11, color='#1E3A5F', y=1.01)

    for nombre, d in esc.items():
        ax1.plot(meses_x, d['rev'], color=COLOR[nombre], lw=lw[nombre],
                 label=nombre.capitalize(), zorder=3)
        ax1.axhline(d['eq_rev'], color=COLOR[nombre], lw=0.8, ls=':', alpha=0.5)
        ax2.plot(meses_x, d['ha'],  color=COLOR[nombre], lw=lw[nombre],
                 label=nombre.capitalize())
        ax2.axhline(d['eq_ha'], color=COLOR[nombre], lw=0.8, ls=':', alpha=0.5,
                    label=f"Eq {nombre}: {d['eq_ha']:,.0f} ha")

    for i, (sn, costo) in enumerate(COSTOS.items()):
        col = COLOR['costo_a'] if i == 0 else COLOR['costo_b']
        ax1.axhline(costo, color=col, lw=1.5, ls='--', label=f'Costo: {sn}', zorder=4)
        for nombre, d in esc.items():
            be = mes_breakeven(d['rev'], costo)
            if be and be <= p['meses']:
                ax1.scatter([be], [costo], color=COLOR[nombre], s=60, zorder=5)

    ax2.axhline(p['techo_ha'], color=COLOR['gris'], lw=1, ls='--',
                label=f"Techo: {p['techo_ha']:,} ha")

    for ax, title, ylabel in [
        (ax1, 'Ingresos mensuales vs Costos Fijos', 'MXN / mes'),
        (ax2, 'Hectareas activas en el tiempo', 'Ha activas'),
    ]:
        ax.set_xlabel('Mes'); ax.set_ylabel(ylabel)
        ax.set_facecolor('#FAFAFA'); ax.spines[['top','right']].set_visible(False)
        ax.legend(fontsize=7.5, ncol=2 if ax==ax1 else 1)
        ax.set_xlim(1, p['meses']); ax.set_ylim(bottom=0)
        ax.set_title(title, fontsize=10, color='#1E3A5F')

    ax1.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}k' if v >= 1000 else f'${v:.0f}'))
    fig.text(0.5, -0.02,
             'Puntos = mes de breakeven. Lineas punteadas = equilibrio teorico.',
             ha='center', fontsize=8, color='#888')
    plt.tight_layout()
    plt.show()


graficar(PARAMS)

## 5. Exploración interactiva con sliders

> Requiere `ipywidgets`. Si no funciona: `pip install ipywidgets` y reinicia el kernel.

In [ ]:
if not WIDGETS_OK:
    print('ipywidgets no disponible. Modifica PARAMS y ejecuta graficar(PARAMS).')
else:
    @interact(
        techo_ha    = widgets.IntSlider(500, 500, 40_000, 500,
                          description='Techo (ha)',
                          style={'description_width': '130px'},
                          layout=widgets.Layout(width='500px')),
        adquisicion = widgets.FloatSlider(3.0, 0.5, 15.0, 0.5,
                          description='Adquisicion (%)',
                          style={'description_width': '130px'},
                          layout=widgets.Layout(width='500px')),
        churn       = widgets.FloatSlider(5.0, 0.5, 20.0, 0.5,
                          description='Churn (%/mes)',
                          style={'description_width': '130px'},
                          layout=widgets.Layout(width='500px')),
        precio_ha   = widgets.IntSlider(120, 60, 300, 10,
                          description='Precio $/ha/ciclo',
                          style={'description_width': '130px'},
                          layout=widgets.Layout(width='500px')),
    )
    def explorar(techo_ha, adquisicion, churn, precio_ha):
        p = {
            'techo_ha':    techo_ha,
            'adquisicion': adquisicion / 100,
            'churn':       churn / 100,
            'precio_ha':   precio_ha,
            'ciclos_anio': 2,
            'meses':       48,
        }
        graficar(p)
        esc = construir_escenarios(p)
        eq_base = esc['base']['eq_ha']
        print(f'Equilibrio base: {eq_base:,.0f} ha ({eq_base/techo_ha*100:.1f}% del techo)')
        for sn, costo in COSTOS.items():
            print(f'  Breakeven {sn}:')
            for nombre, d in esc.items():
                be = mes_breakeven(d['rev'], costo)
                print(f'    {nombre.capitalize():12}: {("Mes " + str(be)) if be else "No alcanza (48m)"}')

## 6. Heatmap de sensibilidad

Muestra el mes de breakeven para cada combinación de adquisición × churn.

In [ ]:
costo_ref  = list(COSTOS.values())[0]  # Stack actual
adq_vals   = np.arange(0.5, 16.0, 1.0) / 100  # 0.5% a 15%
churn_vals = np.arange(0.5, 21.0, 1.0) / 100  # 0.5% a 20%

matriz = np.full((len(churn_vals), len(adq_vals)), 49.0)
for i, ch in enumerate(churn_vals):
    for j, acq in enumerate(adq_vals):
        ha_arr  = simular_adopcion(PARAMS['techo_ha'], acq, ch, PARAMS['meses'])
        rev_arr = ingreso_mensual(ha_arr, PARAMS['precio_ha'], PARAMS['ciclos_anio'])
        be = mes_breakeven(rev_arr, costo_ref)
        if be:
            matriz[i, j] = be

fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
im = ax.imshow(matriz, aspect='auto', origin='lower', cmap='RdYlGn_r', vmin=1, vmax=49)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Mes de breakeven (Stack actual)', fontsize=9)
cbar.ax.set_yticks([1, 12, 24, 36, 49])
cbar.ax.set_yticklabels(['Mes 1', '12', '24', '36', '>48 meses'])

ax.set_xticks(range(len(adq_vals)))
ax.set_xticklabels([f'{v*100:.0f}%' for v in adq_vals], fontsize=7.5, rotation=45)
ax.set_yticks(range(len(churn_vals)))
ax.set_yticklabels([f'{v*100:.0f}%' for v in churn_vals], fontsize=7.5)
ax.set_xlabel('Tasa de adquisicion mensual', fontsize=10)
ax.set_ylabel('Churn mensual', fontsize=10)
ax.set_title(
    f"Mes de breakeven - Stack actual (~$1,613 MXN/mes)\n"
    f"Techo: {PARAMS['techo_ha']:,} ha | Precio: ${PARAMS['precio_ha']} MXN/ha/ciclo",
    fontsize=11, color='#1E3A5F'
)

# FIX: usar argmin en lugar de isclose — el valor exacto puede no estar en el array
# adq_vals = [0.005, 0.015, 0.025...] no contiene exactamente 0.03
idx_adq   = int(np.argmin(np.abs(adq_vals   - PARAMS['adquisicion'])))
idx_churn = int(np.argmin(np.abs(churn_vals - PARAMS['churn'])))
label_be  = f"Params base (adq={PARAMS['adquisicion']*100:.0f}%, churn={PARAMS['churn']*100:.0f}%)"
ax.scatter([idx_adq], [idx_churn], color='white', s=200, zorder=5,
           marker='*', label=label_be)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

print('Verde = breakeven rapido | Rojo = lento o inalcanzable')
print('La estrella blanca es el escenario base actual')

## 7. Conclusiones

1. **El churn es el parámetro crítico**, no el precio. Con 5%/mes, el 46% de los clientes de enero no están en diciembre.

2. **El equilibrio real < techo de mercado.** En el caso base, el negocio se estabiliza en 1,875 ha — el 37.5% de las 5,000 disponibles.

3. **Stack actual: viable con ~80 ha.** El costo de ~$1,613 MXN/mes es bajo. El reto real es Azure (~$24,390 MXN/mes), que requiere ~1,220 ha.

4. **Sin Plan 2 (institucional), el modelo es frágil.** Un módulo DR-041 a $35,000 MXN/año equivale a tener 146 ha gratuitas.

### Próximos pasos
- Calibrar `adquisicion` con datos de adopción real en el módulo
- Agregar estacionalidad (churn mayor fuera de temporada de riego)
- Modelar Plan 2 como ingreso fijo complementario
- Conectar `techo_ha` con el padrón real de usuarios DR-041